# SODE-Guard: rebuttal experiments on ICS3D (Colab)

Runs every experiment the reviewers asked for on **one** released code path:

* training (10 models): SODE-Guard (paper setting), SODE-Guard without the regulariser, certifiable SODE-Guard, SDE-TGNN (re-implementation), a neural ODE (SODE-Guard with the diffusion off), a certifiable neural ODE, an MLP, the MLP with PGD-AT, a Gaussian-noise MLP for randomised smoothing, and a Lipschitz-bounded MLP;
* attacks at ε = 0.03 (ℓ∞, z-scored features): PGD-40, PGD-200 with 5 restarts, EOT-PGD-100, APGD-CE/DLR with EOT, an expected-margin attack, Square (black box) and transfer, scored with **fixed** and **fresh** Brownian paths, plus ε and step sweeps;
* certified accuracy at matched ℓ2: Theorem A (certifiable SODE-Guard), certifiable neural ODE, Lipschitz MLP and randomised smoothing;
* a train/test duplicate (leakage) check written to `meta.json`;
* the near-boundary mass P[G(x) ≤ β] with and without the regulariser.

**Runtime:** set *Runtime → Change runtime type → GPU* (A100 or L4 if available, T4 works but is slower). Expect roughly 1–3 hours in total. Results are written to Google Drive after every stage, so a disconnect loses at most one stage.

**At the end:** download `results.json`, `meta.json` and `run.log` (last cell) and upload them back to the chat.

In [ ]:
!nvidia-smi -L || echo "No GPU: switch the runtime to GPU" 

## 1. Code and dependencies

In [ ]:
%cd /content
!rm -rf CV && git clone -q --depth 1 -b claude/implement-ode-guard-A39oM https://github.com/rogerpanel/CV.git
%cd /content/CV/SODE-Guard
!git log -1 --format='code commit: %h %ci'
!pip -q install kaggle scikit-learn scipy statsmodels pyyaml pandas

## 2. Save results to Google Drive (survives disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/sode_guard_rebuttal'
import os; os.makedirs(OUT, exist_ok=True); print(OUT)

## 3. Kaggle credentials

Either add `KAGGLE_USERNAME` and `KAGGLE_KEY` as Colab secrets (key icon on the left), or upload your `kaggle.json` when prompted.

In [ ]:
import os, json
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print('using Colab secrets')
except Exception:
    from google.colab import files
    up = files.upload()                       # choose kaggle.json
    os.makedirs('/root/.kaggle', exist_ok=True)
    open('/root/.kaggle/kaggle.json', 'wb').write(up['kaggle.json'])
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('using kaggle.json')

## 4. List the ICS3D files

In [ ]:
DATASET = 'rogernickanaedevha/integrated-cloud-security-3datasets-ics3d'
!kaggle datasets files {DATASET} --page-size 200

## 5. Download

Downloading everything (about 8.4 GB) is simplest. If disk is tight, download single files with `-f <name>` instead.

In [ ]:
!mkdir -p /content/ics3d
!kaggle datasets download -d {DATASET} -p /content/ics3d --unzip -q
!find /content/ics3d -name '*.csv' -exec ls -lh {} \; | awk '{print $5, $9}'

## 6. Inspect the CSVs and choose the flow files

ICS3D mixes flow records with other logs. Pick the CSV(s) that contain **per-flow numeric statistics and a class label** (the network-flow parts: container/Kubernetes flows and Edge-IIoT flows). The cell prints each file's size, number of numeric columns and candidate label columns.

In [ ]:
import glob, pandas as pd
cands = ['label','Label','attack_cat','Attack_type','attack_type','Category','category','class','Class','Attack','attack','type']
for f in sorted(glob.glob('/content/ics3d/**/*.csv', recursive=True)):
    try:
        h = pd.read_csv(f, nrows=200, low_memory=False)
    except Exception as e:
        print('unreadable', f, e); continue
    num = h.select_dtypes('number').shape[1]
    labs = [c for c in cands if c in h.columns]
    print(f"{os.path.getsize(f)/1e6:9.1f} MB | {num:3d} numeric cols | labels {labs} | {f}")

Set `CSV_GLOB` to the chosen flow file(s) and `LABEL` to its label column (leave `LABEL = ''` to auto-detect).

In [ ]:
CSV_GLOB = '/content/ics3d/**/CHANGE_ME*.csv'   # <- edit
LABEL = ''                                       # <- e.g. 'Label' or 'Attack_type'
print(glob.glob(CSV_GLOB, recursive=True))

## 7. Prepare (stratified, capped at 20,000 flows per class, 70/15/15, z-scored)

In [ ]:
lab = f"--label-col '{LABEL}'" if LABEL else ''
!python scripts/rebuttal_run.py prepare --csv "{CSV_GLOB}" {lab} --out {OUT} 2>&1 | tee -a {OUT}/run.log
print(open(f'{OUT}/meta.json').read()[:2000])

## 8. Train the ten models (about 10 epochs each)

In [ ]:
!python scripts/rebuttal_run.py train --out {OUT} 2>&1 | tee -a {OUT}/run.log

## 9. Attack suite (the longest stage)

In [ ]:
!python scripts/rebuttal_run.py attack --out {OUT} 2>&1 | tee -a {OUT}/run.log

## 10. Certification and regulariser measurement

In [ ]:
!python scripts/rebuttal_run.py certify --out {OUT} 2>&1 | tee -a {OUT}/run.log
!python scripts/rebuttal_run.py regulariser --out {OUT} 2>&1 | tee -a {OUT}/run.log

## 11. Summary and download

In [ ]:
!python scripts/rebuttal_run.py report --out {OUT} 2>&1 | tee {OUT}/summary.txt
from google.colab import files
for f in ['results.json', 'meta.json', 'run.log', 'summary.txt']:
    files.download(f'{OUT}/{f}')

### If time is short
Run step 9 with fewer flows and fewer EOT samples, e.g.
`--n-attack 500 --eot 16 --restarts 3 --square-queries 2000`, and train with `--epochs 5`.

## 12. Optional, if time remains: temporal split

Reviewer FiAv asks for a temporal split. This only works if the chosen CSV has a timestamp column (the prepare step stops with a message otherwise). It trains three models and runs the cheaper attacks.

In [ ]:
OUT_T = OUT + '_temporal'
!python scripts/rebuttal_run.py prepare --csv "{CSV_GLOB}" {lab} --split temporal --out {OUT_T} 2>&1 | tee -a {OUT_T}.log
!python scripts/rebuttal_run.py train --out {OUT_T} --models sode sdetgnn mlp 2>&1 | tee -a {OUT_T}.log
!python scripts/rebuttal_run.py attack --out {OUT_T} --attack-models sode sdetgnn mlp --eot 16 --restarts 3 --square-queries 2000 --step-sweep 40 200 --eps-sweep 0.03 0.3 2>&1 | tee -a {OUT_T}.log
!python scripts/rebuttal_run.py report --out {OUT_T} 2>&1 | tee {OUT_T}_summary.txt
for f in [f'{OUT_T}/results.json', f'{OUT_T}/meta.json', f'{OUT_T}_summary.txt']:
    files.download(f)